# Chest X-ray Classification with Transfer Learning and Grad-CAM

Master’s Deep Learning Applications coursework. Classes: COVID, NORMAL, PNEUMONIA.

This notebook retains historical code and textual outputs from the local Milestone3 notebook. Narrative was shortened to remove conflicting results and unsupported claims. Embedded image outputs were removed pending redistribution review. Outputs are historical and have not been regenerated after packaging. See ../docs/results.md.

Run in Google Colab with your dataset mounted in Drive. For local Jupyter, replace the Drive mount and final download cells with local paths. Educational research only; not for medical decisions.


## 1. Install Dependencies


In [ ]:
!pip install torch torchvision timm --quiet
!pip install scikit-learn matplotlib seaborn numpy Pillow tqdm opencv-python-headless --quiet
!pip install gradio --quiet
print('All packages installed successfully!')


All packages installed successfully!


## 2. Import Libraries


In [ ]:
import os
import json
import copy
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    f1_score, accuracy_score,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
import cv2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Using device: cuda
GPU: Tesla T4


## 3. Configuration


In [ ]:
DATA_DIR   = './data'      # <── CHANGE THIS: folder containing COVID/, NORMAL/, PNEUMONIA/
MODEL_DIR  = 'models'
OUTPUT_DIR = 'outputs'
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

IMG_SIZE      = 224                           # Standard ImageNet input resolution
IMAGENET_MEAN = [0.485, 0.456, 0.406]         # ImageNet channel means
IMAGENET_STD  = [0.229, 0.224, 0.225]         # ImageNet channel std devs

BATCH_SIZE    = 32
EPOCHS        = 25
LR            = 1e-4      # Conservative LR for fine-tuning pretrained weights
LR_BASELINE   = 1e-3      # Higher LR for from-scratch baseline training
WEIGHT_DECAY  = 1e-4
PATIENCE      = 5         # Early stopping patience
LR_T_MAX      = 25        # CosineAnnealingLR period
NUM_CLASSES   = 3

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

PALETTE = {
    'covid':     '#E74C3C',
    'normal':    '#2ECC71',
    'pneumonia': '#3498DB',
    'neutral':   '#6B7280',
}
plt.rcParams.update({
    'font.family':     'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})
FIGURE_DPI = 150

print('Configuration set.')
print(f'  Data dir  : {DATA_DIR}')
print(f'  Output dir: {OUTPUT_DIR}')
print(f'  Device    : {DEVICE}')
print(f'  Seed      : {RANDOM_SEED}')


Configuration set.
  Data dir  : https://drive.google.com/drive/mydrive/Dataset_Pneumonia
  Output dir: outputs
  Device    : cuda
  Seed      : 42


## 4. Upload / Mount Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/Dataset_Pneumonia'  # <── change path

print(f'Dataset path : {DATA_DIR}')
print(f'Contents     : {os.listdir(DATA_DIR)}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset path : /content/drive/MyDrive/Dataset_Pneumonia
Contents     : ['PNEUMONIA', '.DS_Store', 'NORMAL', 'COVID']


## 5. Data Preparation


In [ ]:
CLASS_NAMES = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
print(f'Classes found: {CLASS_NAMES}')

print('\n=== Missing / Corrupted File Check ===')
class_counts = {}
total_corrupt = 0
for cls in CLASS_NAMES:
    cls_path = os.path.join(DATA_DIR, cls)
    valid, corrupt = 0, 0
    for fname in os.listdir(cls_path):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(os.path.join(cls_path, fname))
                img.verify()   # checks file integrity without decoding fully
                valid += 1
            except Exception:
                corrupt += 1
                print(f'  WARNING: corrupted file — {fname}')
    class_counts[cls] = valid
    total_corrupt += corrupt
    print(f'  {cls:<12}: {valid:>5} valid  |  {corrupt} corrupted')

total = sum(class_counts.values())
print(f'  {"TOTAL":<12}: {total:>5} images')
print(f'\nMissing/Corrupted: {total_corrupt}')
print(f'Quality check    : {"PASSED ✓" if total_corrupt == 0 else "ISSUES FOUND ✗"}')

print('\n=== Class Balance Check ===')
for cls, count in class_counts.items():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'  {cls:<12}: {count:>5}  ({pct:.1f}%)  {bar}')
print('Review the class counts above before selecting imbalance handling.')


In [ ]:
print('=== Outlier Detection — Pixel Intensity Statistics (50-image sample per class) ===')
print(f'{"Class":<12} {"Min":>8} {"Max":>8} {"Mean":>8} {"Std":>8}')
print('-' * 50)
for cls in CLASS_NAMES:
    cls_path   = os.path.join(DATA_DIR, cls)
    all_files  = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    sample     = random.sample(all_files, min(50, len(all_files)))
    pixels     = []
    for fname in sample:
        arr = np.array(Image.open(os.path.join(cls_path, fname)).convert('RGB'))
        pixels.extend(arr.flatten().tolist())
    px = np.array(pixels)
    print(f'{cls:<12} {px.min():>8.1f} {px.max():>8.1f} {px.mean():>8.1f} {px.std():>8.1f}')
print('Sample statistics only; this does not certify every image is free of outliers.')


In [ ]:

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print('Transforms defined:')
print('  Training set  : Resize + 4 augmentations + ToTensor + Normalise  (7 steps)')
print('  Val / Test set: Resize + ToTensor + Normalise                     (3 steps)')


Transforms defined:
  Training set  : Resize + 4 augmentations + ToTensor + Normalise  (7 steps)
  Val / Test set: Resize + ToTensor + Normalise                     (3 steps)


In [ ]:

full_dataset = ImageFolder(root=DATA_DIR, transform=train_transforms)
CLASS_TO_IDX = full_dataset.class_to_idx
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
print(f'Class → index mapping: {CLASS_TO_IDX}')

total_size  = len(full_dataset)
train_size  = int(TRAIN_RATIO * total_size)
val_size    = int(VAL_RATIO   * total_size)
test_size   = total_size - train_size - val_size

generator = torch.Generator().manual_seed(RANDOM_SEED)
train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)

val_ds.dataset  = ImageFolder(root=DATA_DIR, transform=val_test_transforms)
test_ds.dataset = ImageFolder(root=DATA_DIR, transform=val_test_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nDataset split (seed={RANDOM_SEED}):')
print(f'  Training  : {train_size:>5} images  (70%)')
print(f'  Validation: {val_size:>5} images  (15%)')
print(f'  Test      : {test_size:>5} images  (15%)  ← held out until final evaluation')


Class → index mapping: {'COVID': 0, 'NORMAL': 1, 'PNEUMONIA': 2}

Dataset split (seed=42):
  Training  :  3659 images  (70%)
  Validation:   784 images  (15%)
  Test      :   785 images  (15%)  ← held out until final evaluation


## 6. Exploratory Data Analysis


In [ ]:
colors_list = [PALETTE['covid'], PALETTE['normal'], PALETTE['pneumonia']]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(class_counts.keys(), class_counts.values(), color=colors_list, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Number of Images')
for i, (cls, count) in enumerate(class_counts.items()):
    axes[0].text(i, count + 15, str(count), ha='center', fontweight='bold', fontsize=11)

axes[1].pie(
    class_counts.values(), labels=class_counts.keys(),
    colors=colors_list, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11}
)
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')

plt.suptitle('Figure 1: Dataset Overview — 5,228 Chest X-Ray Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig1_class_distribution.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Note: Near-equal class balance (31–35%) — no oversampling required.')


Note: Near-equal class balance (31–35%) — no oversampling required.


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for row, (cls, color) in enumerate(zip(CLASS_NAMES, colors_list)):
    cls_path = os.path.join(DATA_DIR, cls)
    all_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    samples   = random.sample(all_files, 5)
    for col, fname in enumerate(samples):
        img = Image.open(os.path.join(cls_path, fname)).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls, fontsize=12, fontweight='bold', color=color, pad=6)

plt.suptitle('Figure 2: Sample Chest X-Ray Images — 5 per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig2_sample_images.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()


In [ ]:
def denorm(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, ax in enumerate(axes.flatten()):
    if i < 16:
        img = denorm(imgs[i]).permute(1, 2, 0).numpy()
        ax.imshow(img); ax.axis('off')
        ax.set_title(IDX_TO_CLASS[labels[i].item()], fontsize=8)

plt.suptitle('Figure 3: Augmented Training Samples (flip, rotate, jitter, affine visible)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig3_augmented_samples.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()


## 7. Baseline Model — Custom CNN (From Scratch)


In [ ]:
class BaseCNN(nn.Module):
    """
    Baseline Custom CNN — no pretrained weights.

    Architecture (4 convolutional blocks + classifier head):
      Block 1: Conv2d(3→32,   3×3) → BatchNorm → ReLU → MaxPool(2×2)
      Block 2: Conv2d(32→64,  3×3) → BatchNorm → ReLU → MaxPool(2×2)
      Block 3: Conv2d(64→128, 3×3) → BatchNorm → ReLU → MaxPool(2×2)
      Block 4: Conv2d(128→256,3×3) → BatchNorm → ReLU → MaxPool(2×2)
      Head: AdaptiveAvgPool(4×4) → Flatten → FC(4096→512) → ReLU → Dropout(0.5)
                                            → FC(512→128) → ReLU → Dropout(0.3)
                                            → FC(128→3)

    Design rationale:
      - Doubling filter depth per block (32→64→128→256): each block learns
        increasingly abstract features — low-level edges → textures → anatomy.
      - BatchNorm: normalises activations per batch, accelerating convergence
        and reducing sensitivity to learning rate choice.
      - Progressive Dropout (0.5→0.3): stronger regularisation early in the
        classifier, reduced closer to the output layer.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super(BaseCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32,  64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
            nn.Linear(256*4*4, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, 128),     nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

baseline_model = BaseCNN().to(DEVICE)
p_baseline = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
print(f'Baseline CNN — Trainable parameters: {p_baseline:,}')


Baseline CNN — Trainable parameters: 2,553,091


## 8. Transfer Learning Model Architectures


In [ ]:

def build_resnet50(freeze_backbone=False):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.4),
        nn.Linear(512, NUM_CLASSES)
    )
    return model

resnet_model = build_resnet50(freeze_backbone=False).to(DEVICE)
p_resnet = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'ResNet50 (full fine-tune) — Trainable parameters: {p_resnet:,}')



def build_efficientnet_b0():
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
    return model

efficientnet_model = build_efficientnet_b0().to(DEVICE)
p_eff = sum(p.numel() for p in efficientnet_model.parameters() if p.requires_grad)
print(f'EfficientNet-B0 — Trainable parameters: {p_eff:,}')
print(f'\nParameter efficiency vs ResNet50: EfficientNet-B0 uses {p_resnet/p_eff:.1f}x fewer parameters.')


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 204MB/s]


ResNet50 (full fine-tune) — Trainable parameters: 24,558,659


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

EfficientNet-B0 — Trainable parameters: 4,011,391

Parameter efficiency vs ResNet50: EfficientNet-B0 uses 6.1x fewer parameters.


## 9. PyTorch DataLoaders — Confirmed Above (Section 5)


## 10. Model Training


In [ ]:
def run_epoch(model, loader, criterion, optimiser, training):
    model.train(training)
    total_loss = correct = total = 0
    with torch.set_grad_enabled(training):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            if training:
                optimiser.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimiser.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total


def train_model(model, name, train_loader, val_loader, lr=LR, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss()                          # multi-class cross-entropy
    optimiser = optim.AdamW(                                   # decoupled weight decay
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=LR_T_MAX, eta_min=1e-6)
    ckpt_path = os.path.join(MODEL_DIR, f'best_{name}.pt')

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'best_epoch': 0}
    best_val_acc  = 0.0
    patience_ctr  = 0

    SEP = '-' * 68
    HDR = f'{"Epoch":>6} {"T-Loss":>8} {"T-Acc":>7} {"V-Loss":>8} {"V-Acc":>7} {"LR":>10}'
    print(f'\n{"="*68}')
    print(f' Training: {name}  |  LR={lr}  |  Optimiser: AdamW  |  Loss: CrossEntropyLoss')
    print(f'{"="*68}')
    print(SEP)
    print(HDR)
    print(SEP)

    for epoch in range(1, epochs + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimiser, True)
        vl, va = run_epoch(model, val_loader,   criterion, None,      False)
        scheduler.step()
        lr_now = optimiser.param_groups[0]['lr']

        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['train_acc'].append(ta)
        history['val_acc'].append(va)

        saved = ''
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), ckpt_path)
            patience_ctr = 0
            history['best_epoch'] = epoch
            saved = '  ✓'
        else:
            patience_ctr += 1

        print(f'{epoch:>6} {tl:>8.4f} {ta:>6.1%} {vl:>8.4f} {va:>6.1%} {lr_now:>10.2e}{saved}')

        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping triggered at epoch {epoch} (patience={PATIENCE}).')
            break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print(f'\nBest checkpoint restored from epoch {history["best_epoch"]}.')
    print(f'Best Validation Accuracy: {best_val_acc:.4f}  ({best_val_acc*100:.2f}%)')
    return history, best_val_acc

print('Training functions defined ✓')


Training functions defined ✓


In [ ]:
baseline_history, baseline_best = train_model(
    baseline_model, 'baseline_cnn',
    train_loader, val_loader, lr=LR_BASELINE
)



 Training: baseline_cnn  |  LR=0.001  |  Optimiser: AdamW  |  Loss: CrossEntropyLoss
--------------------------------------------------------------------
 Epoch   T-Loss   T-Acc   V-Loss   V-Acc         LR
--------------------------------------------------------------------
     1   0.5792  75.2%   0.3099  90.2%   9.96e-04  ✓
     2   0.3863  86.3%   0.3039  89.9%   9.84e-04
     3   0.3357  88.5%   0.2210  93.5%   9.65e-04  ✓
     4   0.3052  89.6%   0.4398  83.7%   9.38e-04
     5   0.2975  89.0%   0.3133  90.9%   9.05e-04
     6   0.2483  91.4%   0.2396  93.1%   8.65e-04
     7   0.2571  91.4%   0.2598  92.5%   8.19e-04
     8   0.2333  92.2%   0.2734  93.0%   7.68e-04

Early stopping triggered at epoch 8 (patience=5).

Best checkpoint restored from epoch 3.
Best Validation Accuracy: 0.9349  (93.49%)


In [ ]:
resnet_history, resnet_best = train_model(
    resnet_model, 'resnet50',
    train_loader, val_loader, lr=LR
)



 Training: resnet50  |  LR=0.0001  |  Optimiser: AdamW  |  Loss: CrossEntropyLoss
--------------------------------------------------------------------
 Epoch   T-Loss   T-Acc   V-Loss   V-Acc         LR
--------------------------------------------------------------------
     1   0.3475  86.4%   0.1312  95.9%   9.96e-05  ✓
     2   0.1647  94.8%   0.0823  97.4%   9.84e-05  ✓
     3   0.1021  96.7%   0.0933  97.6%   9.65e-05  ✓
     4   0.0961  96.9%   0.1140  96.3%   9.39e-05
     5   0.0805  97.7%   0.0846  97.3%   9.05e-05
     6   0.0700  97.7%   0.0560  98.7%   8.66e-05  ✓
     7   0.0496  98.4%   0.0979  97.6%   8.21e-05
     8   0.0394  98.6%   0.2020  95.7%   7.70e-05
     9   0.0385  98.6%   0.0861  98.1%   7.16e-05
    10   0.0279  99.1%   0.0795  98.3%   6.58e-05
    11   0.0294  99.2%   0.0692  98.7%   5.98e-05

Early stopping triggered at epoch 11 (patience=5).

Best checkpoint restored from epoch 6.
Best Validation Accuracy: 0.9872  (98.72%)


In [ ]:
efficientnet_history, eff_best = train_model(
    efficientnet_model, 'efficientnet_b0',
    train_loader, val_loader, lr=LR
)



 Training: efficientnet_b0  |  LR=0.0001  |  Optimiser: AdamW  |  Loss: CrossEntropyLoss
--------------------------------------------------------------------
 Epoch   T-Loss   T-Acc   V-Loss   V-Acc         LR
--------------------------------------------------------------------
     1   0.4995  88.1%   0.3328  91.8%   9.96e-05  ✓
     2   0.2159  94.1%   0.3152  94.0%   9.84e-05  ✓
     3   0.1280  96.6%   0.1954  96.4%   9.65e-05  ✓
     4   0.0998  97.4%   0.1774  97.2%   9.39e-05  ✓
     5   0.0836  98.0%   0.2198  96.4%   9.05e-05
     6   0.0521  98.6%   0.2568  95.9%   8.66e-05
     7   0.0571  98.2%   0.2044  97.6%   8.21e-05  ✓
     8   0.0399  98.9%   0.1634  98.2%   7.70e-05  ✓
     9   0.0392  98.8%   0.1789  97.3%   7.16e-05
    10   0.0266  99.0%   0.1968  97.4%   6.58e-05
    11   0.0190  99.4%   0.2149  97.6%   5.98e-05
    12   0.0181  99.3%   0.1898  97.7%   5.36e-05
    13   0.0196  99.5%   0.1721  98.0%   4.74e-05

Early stopping triggered at epoch 13 (patience=5).


## 11. Training Curves


In [ ]:
all_histories = {
    'Baseline CNN':    (baseline_history,    '#E74C3C'),
    'ResNet50':        (resnet_history,       '#3498DB'),
    'EfficientNet-B0': (efficientnet_history, '#2ECC71'),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
best_epochs = {}

for name, (hist, color) in all_histories.items():
    ep   = range(1, len(hist['train_acc']) + 1)
    best = hist.get('best_epoch', len(ep))
    best_epochs[name] = best

    ax1.plot(ep, hist['val_acc'],   '-',  color=color, lw=2.5, label=f'{name} (val)')
    ax1.plot(ep, hist['train_acc'], '--', color=color, lw=1.5, alpha=0.5)
    ax1.axvline(best, color=color, lw=1, linestyle=':', alpha=0.7)

    ax2.plot(ep, hist['val_loss'],   '-',  color=color, lw=2.5, label=f'{name} (val)')
    ax2.plot(ep, hist['train_loss'], '--', color=color, lw=1.5, alpha=0.5)
    ax2.axvline(best, color=color, lw=1, linestyle=':', alpha=0.7)

for ax, ylabel, title in zip(
    [ax1, ax2],
    ['Accuracy', 'Loss'],
    ['Validation Accuracy (solid=val, dashed=train)', 'Validation Loss (solid=val, dashed=train)']
):
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.suptitle('Figure 4: Training Curves — All Models  |  Dotted lines = best checkpoint epoch',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig4_training_curves.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()


## 12. Model Evaluation


In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE)
        out    = model(imgs)
        probs  = torch.softmax(out, dim=1)
        preds  = out.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

class_names_list = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

print('=== EfficientNet-B0 — Test Set Evaluation ===')
preds, labels_gt, probs = get_predictions(efficientnet_model, test_loader)

acc = accuracy_score(labels_gt, preds)
f1  = f1_score(labels_gt, preds, average='weighted')
labels_bin = label_binarize(labels_gt, classes=list(range(NUM_CLASSES)))
auc_score  = roc_auc_score(labels_bin, probs, average='macro', multi_class='ovr')

print(f'\nTest Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'Weighted F1    : {f1:.4f}')
print(f'Macro ROC-AUC  : {auc_score:.4f}')
print(f'\nClassification Report:')
print(classification_report(labels_gt, preds, target_names=class_names_list))


=== EfficientNet-B0 — Test Set Evaluation ===

Test Accuracy  : 0.9783  (97.83%)
Weighted F1    : 0.9784
Macro ROC-AUC  : 0.9974

Classification Report:
              precision    recall  f1-score   support

       COVID       1.00      0.99      1.00       229
      NORMAL       0.95      0.99      0.97       270
   PNEUMONIA       0.99      0.96      0.97       286

    accuracy                           0.98       785
   macro avg       0.98      0.98      0.98       785
weighted avg       0.98      0.98      0.98       785



In [ ]:
cm = confusion_matrix(labels_gt, preds)
pct = cm / cm.sum() * 100
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names_list, yticklabels=class_names_list,
    linewidths=0.5, ax=ax, annot_kws={'size': 13}
)
ax.set_title('Figure 5: Confusion Matrix — EfficientNet-B0 (Test Set)', fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig5_confusion_matrix.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = [PALETTE['covid'], PALETTE['normal'], PALETTE['pneumonia']]

for i, (cls_name, color) in enumerate(zip(class_names_list, colors_roc)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], probs[:, i])
    cls_auc = roc_auc_score(labels_bin[:, i], probs[:, i])
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{cls_name} (AUC = {cls_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_title('Figure 6: ROC Curves — EfficientNet-B0 (Test Set)', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig6_roc_curves.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()


## 13. Results Summary Table


In [ ]:
import pandas as pd

def quick_eval(model, loader):
    p, l, pr = get_predictions(model, loader)
    lb = label_binarize(l, classes=list(range(NUM_CLASSES)))
    return (
        round(accuracy_score(l, p), 4),
        round(f1_score(l, p, average='weighted'), 4),
        round(roc_auc_score(lb, pr, average='macro', multi_class='ovr'), 4),
    )

b_acc, b_f1, b_auc = quick_eval(baseline_model,    test_loader)
r_acc, r_f1, r_auc = quick_eval(resnet_model,       test_loader)
e_acc, e_f1, e_auc = quick_eval(efficientnet_model, test_loader)

results_summary = pd.DataFrame([
    {'Model': 'Baseline CNN (from scratch)', 'Params': '2.5M',  'Test Accuracy': b_acc, 'Weighted F1': b_f1, 'Macro AUC': b_auc},
    {'Model': 'ResNet50 (fine-tuned)',        'Params': '24.6M', 'Test Accuracy': r_acc, 'Weighted F1': r_f1, 'Macro AUC': r_auc},
    {'Model': 'EfficientNet-B0 (proposed)',   'Params': '4.0M',  'Test Accuracy': e_acc, 'Weighted F1': e_f1, 'Macro AUC': e_auc},
])

print('='*70)
print('  TABLE 1: Model Comparison — Test Set')
print('='*70)
print(results_summary.to_string(index=False))
print('='*70)
print('\nNote: EfficientNet-B0 achieves near-equivalent accuracy to ResNet50')
print('with approximately 6.1x fewer parameters.')


  TABLE 1: Model Comparison — Test Set
                      Model Params  Test Accuracy  Weighted F1  Macro AUC
Baseline CNN (from scratch)   2.5M         0.9350       0.9348     0.9842
      ResNet50 (fine-tuned)  24.6M         0.9783       0.9784     0.9984
 EfficientNet-B0 (proposed)   4.0M         0.9783       0.9784     0.9974

Note: EfficientNet-B0 achieves near-equivalent accuracy to ResNet50
with 6.1x fewer parameters — optimal for deployment.


## 13b. Ablation Study


In [ ]:
print('--- Ablation 1: ResNet50 FROZEN backbone (feature extraction only) ---')
resnet_frozen = build_resnet50(freeze_backbone=True).to(DEVICE)
frozen_p = sum(p.numel() for p in resnet_frozen.parameters() if p.requires_grad)
print(f'Trainable parameters (frozen): {frozen_p:,}  (only classification head)')
frozen_history, frozen_best = train_model(
    resnet_frozen, 'resnet50_frozen',
    train_loader, val_loader, lr=1e-3, epochs=15
)


--- Ablation 1: ResNet50 FROZEN backbone (feature extraction only) ---
Trainable parameters (frozen): 1,050,627  (only classification head)

 Training: resnet50_frozen  |  LR=0.001  |  Optimiser: AdamW  |  Loss: CrossEntropyLoss
--------------------------------------------------------------------
 Epoch   T-Loss   T-Acc   V-Loss   V-Acc         LR
--------------------------------------------------------------------
     1   0.3808  84.9%   0.1970  93.2%   9.96e-04  ✓
     2   0.2255  91.7%   0.1997  93.2%   9.84e-04
     3   0.2221  92.3%   0.1633  95.2%   9.65e-04  ✓
     4   0.1959  92.8%   0.1642  94.8%   9.38e-04
     5   0.1906  93.0%   0.1495  95.2%   9.05e-04
     6   0.1898  92.8%   0.1835  94.0%   8.65e-04
     7   0.1888  93.1%   0.1480  95.9%   8.19e-04  ✓
     8   0.1805  93.5%   0.1472  95.5%   7.68e-04
     9   0.1589  94.3%   0.1443  95.8%   7.13e-04
    10   0.1602  94.2%   0.1393  96.3%   6.55e-04  ✓
    11   0.1556  94.5%   0.1430  95.2%   5.94e-04
    12   0.1603  94

In [ ]:
print('--- Ablation 2: ResNet50 WITHOUT data augmentation ---')
no_aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])
no_aug_ds    = ImageFolder(root=DATA_DIR, transform=no_aug_transform)
gen          = torch.Generator().manual_seed(RANDOM_SEED)
na_tr, na_vl, _ = random_split(no_aug_ds, [train_size, val_size, test_size], generator=gen)
na_train_loader  = DataLoader(na_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
na_val_loader    = DataLoader(na_vl, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

resnet_noaug = build_resnet50(freeze_backbone=False).to(DEVICE)
noaug_history, noaug_best = train_model(
    resnet_noaug, 'resnet50_no_aug',
    na_train_loader, na_val_loader, lr=LR, epochs=15
)


--- Ablation 2: ResNet50 WITHOUT data augmentation ---

 Training: resnet50_no_aug  |  LR=0.0001  |  Optimiser: AdamW  |  Loss: CrossEntropyLoss
--------------------------------------------------------------------
 Epoch   T-Loss   T-Acc   V-Loss   V-Acc         LR
--------------------------------------------------------------------
     1   0.3327  88.3%   0.0705  98.0%   9.96e-05  ✓
     2   0.0658  98.0%   0.0922  97.4%   9.84e-05
     3   0.0401  98.9%   0.0579  98.7%   9.65e-05  ✓
     4   0.0122  99.6%   0.0821  98.1%   9.39e-05
     5   0.0085  99.7%   0.0621  98.9%   9.05e-05  ✓
     6   0.0061  99.8%   0.0546  99.4%   8.66e-05  ✓
     7   0.0045  99.9%   0.0803  98.9%   8.21e-05
     8   0.0110  99.7%   0.0980  98.7%   7.70e-05
     9   0.0058  99.8%   0.0841  99.1%   7.16e-05
    10   0.0034  99.9%   0.1112  98.7%   6.58e-05
    11   0.0025  99.9%   0.1382  98.7%   5.98e-05

Early stopping triggered at epoch 11 (patience=5).

Best checkpoint restored from epoch 6.
Best Valida

In [ ]:
import pandas as pd

abl_data = [
    {'Configuration': 'Baseline CNN (no pretraining)',    'Val Accuracy': baseline_best},
    {'Configuration': 'ResNet50 — Frozen backbone',       'Val Accuracy': frozen_best},
    {'Configuration': 'ResNet50 — No augmentation',       'Val Accuracy': noaug_best},
    {'Configuration': 'ResNet50 — Full fine-tune + Aug',  'Val Accuracy': resnet_best},
    {'Configuration': 'EfficientNet-B0 (proposed)',       'Val Accuracy': eff_best},
]
abl_df = pd.DataFrame(abl_data)

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ['#6B7280', '#DC2626', '#F59E0B', '#3B82F6', '#16A34A']
bars = ax.barh(abl_df['Configuration'], abl_df['Val Accuracy'],
               color=bar_colors, edgecolor='black', linewidth=0.6, height=0.6)

for bar, val in zip(bars, abl_df['Val Accuracy']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}  ({val*100:.2f}%)', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0.92, 1.01)
ax.set_xlabel('Validation Accuracy', fontsize=11)
ax.set_title('Figure 7: Ablation Study — Contribution of Each Pipeline Component', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig7_ablation_study.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

print('\nTABLE 2: Ablation Study Results')
print(abl_df.to_string(index=False))



TABLE 2: Ablation Study Results
                  Configuration  Val Accuracy
  Baseline CNN (no pretraining)      0.934949
     ResNet50 — Frozen backbone      0.963010
     ResNet50 — No augmentation      0.993622
ResNet50 — Full fine-tune + Aug      0.987245
     EfficientNet-B0 (proposed)      0.982143


## 14. Grad-CAM Explainability (XAI)


In [ ]:
class GradCAM:
    """
    Grad-CAM implementation for EfficientNet-B0.
    Reference: Selvaraju et al., 'Grad-CAM: Visual Explanations from Deep
    Networks via Gradient-based Localization', ICCV 2017.

    Algorithm:
      1. Forward pass → capture feature maps A at target layer
      2. Backward pass for class c → capture gradients ∂y^c/∂A
      3. Global average pool gradients → importance weights α_k^c
      4. Weighted sum of feature maps → class activation map
      5. ReLU → keep only positive activations (those that increase class score)
      6. Normalise to [0, 1] for visualisation
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.grads = None
        self.acts  = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'acts', o.detach())
        )
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'grads', go[0].detach())
        )

    def generate(self, tensor, cls=None):
        self.model.eval()
        out = self.model(tensor)
        cls = out.argmax(1).item() if cls is None else cls
        self.model.zero_grad()
        out[0, cls].backward()
        weights = self.grads.mean(dim=(2, 3), keepdim=True)   # α_k^c
        cam     = torch.relu((weights * self.acts).sum(1)).squeeze().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, cls

grad_cam = GradCAM(efficientnet_model, efficientnet_model.conv_head)
print('Grad-CAM initialised on conv_head layer of EfficientNet-B0 ✓')


Grad-CAM initialised on conv_head layer of EfficientNet-B0 ✓


In [ ]:

fig, axes = plt.subplots(NUM_CLASSES, 6, figsize=(22, 3 * NUM_CLASSES + 2))

samples = {i: [] for i in range(NUM_CLASSES)}
for idx in range(len(test_ds)):
    img_t, lbl = test_ds[idx]
    if len(samples[lbl]) < 3:
        samples[lbl].append((img_t, lbl))
    if all(len(v) == 3 for v in samples.values()):
        break

for row in range(NUM_CLASSES):
    for col, (img_t, lbl) in enumerate(samples[row]):
        inp  = img_t.unsqueeze(0).to(DEVICE).requires_grad_(True)
        cam, pred = grad_cam.generate(inp)

        orig    = denorm(img_t).permute(1, 2, 0).numpy()
        hm      = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        overlay = np.clip(0.6 * orig + 0.4 * plt.cm.jet(hm)[..., :3], 0, 1)

        axes[row, col*2].imshow(orig)
        axes[row, col*2].axis('off')
        axes[row, col*2].set_title(f'True: {IDX_TO_CLASS[lbl]}', fontsize=8)

        axes[row, col*2+1].imshow(overlay)
        axes[row, col*2+1].axis('off')
        col_text = 'green' if pred == lbl else 'red'
        axes[row, col*2+1].set_title(f'Pred: {IDX_TO_CLASS[pred]}', fontsize=8, color=col_text)

plt.suptitle(
    'Figure 8: Grad-CAM Heatmaps — Left: Original X-Ray | Right: Model Attention\n'
    'Illustrative model attribution; not clinical validation',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig8_gradcam.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()


## 15. Save Models and Artefacts


In [ ]:
import pickle

torch.save(baseline_model.state_dict(),     os.path.join(MODEL_DIR, 'baseline_cnn.pt'))
torch.save(resnet_model.state_dict(),       os.path.join(MODEL_DIR, 'resnet50.pt'))
torch.save(efficientnet_model.state_dict(), os.path.join(MODEL_DIR, 'efficientnet_b0.pt'))

config = {
    'model':         'efficientnet_b0',
    'class_to_idx':  CLASS_TO_IDX,
    'idx_to_class':  {str(k): v for k, v in IDX_TO_CLASS.items()},
    'img_size':      IMG_SIZE,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std':  IMAGENET_STD,
    'test_accuracy': float(acc),
    'test_f1':       float(f1),
    'test_auc':      float(auc_score),
    'best_model':    'efficientnet_b0',
}
with open(os.path.join(MODEL_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('All artefacts saved to models/')
print(f'  baseline_cnn.pt       — {os.path.getsize(os.path.join(MODEL_DIR, "baseline_cnn.pt"))/1e6:.1f} MB')
print(f'  resnet50.pt           — {os.path.getsize(os.path.join(MODEL_DIR, "resnet50.pt"))/1e6:.1f} MB')
print(f'  efficientnet_b0.pt    — {os.path.getsize(os.path.join(MODEL_DIR, "efficientnet_b0.pt"))/1e6:.1f} MB')
print(f'  config.json')


All artefacts saved to models/
  baseline_cnn.pt       — 10.2 MB
  resnet50.pt           — 98.6 MB
  efficientnet_b0.pt    — 16.3 MB
  config.json


## 16. Final Results Summary


In [ ]:
print('=' * 65)
print('  FINAL RESULTS SUMMARY')
print('=' * 65)

print('\n--- Table 1: Model Comparison (Validation Accuracy) ---')
print(f'  {"Baseline CNN":<40} {baseline_best:.4f}  ({baseline_best*100:.2f}%)')
print(f'  {"ResNet50 (fine-tuned)":<40} {resnet_best:.4f}  ({resnet_best*100:.2f}%)')
print(f'  {"EfficientNet-B0 (proposed)":<40} {eff_best:.4f}  ({eff_best*100:.2f}%)')

print('\n--- Table 2: Ablation Study (Validation Accuracy) ---')
for row in abl_data:
    print(f'  {row["Configuration"]:<45} {row["Val Accuracy"]:.4f}')

print('\n--- Table 3: EfficientNet-B0 — Test Set ---')
print(f'  Test Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Weighted F1    : {f1:.4f}')
print(f'  Macro ROC-AUC  : {auc_score:.4f}')
print('=' * 65)


  FINAL RESULTS SUMMARY

--- Table 1: Model Comparison (Validation Accuracy) ---
  Baseline CNN                             0.9349  (93.49%)
  ResNet50 (fine-tuned)                    0.9872  (98.72%)
  EfficientNet-B0 (proposed)               0.9821  (98.21%)

--- Table 2: Ablation Study (Validation Accuracy) ---
  Baseline CNN (no pretraining)                 0.9349
  ResNet50 — Frozen backbone                    0.9630
  ResNet50 — No augmentation                    0.9936
  ResNet50 — Full fine-tune + Aug               0.9872
  EfficientNet-B0 (proposed)                    0.9821

--- Table 3: EfficientNet-B0 — Test Set ---
  Test Accuracy  : 0.9783  (97.83%)
  Weighted F1    : 0.9784
  Macro ROC-AUC  : 0.9974


In [ ]:
covid_idx = CLASS_TO_IDX.get('COVID', CLASS_TO_IDX.get('COVID19', 0))

covid_probs   = probs[:, covid_idx]
covid_true    = (labels_gt == covid_idx).astype(int)
quartiles     = np.percentile(covid_probs, [25, 50, 75])
quartile_bins = np.digitize(covid_probs, quartiles)

print('=== Confidence Distribution Check — COVID-19 Class ===')
print(f'{"Quartile":<12} {"N Samples":>10} {"Actual COVID Rate":>18} {"Mean Confidence":>17}')
print('-' * 60)
for q in range(4):
    mask = quartile_bins == q
    if mask.sum() > 0:
        print(f'Q{q+1:<11} {mask.sum():>10} {covid_true[mask].mean():>18.3f} {covid_probs[mask].mean():>17.3f}')

print('\nNote: Demographic metadata unavailable — full fairness audit not possible.')
print('A production system should include age, sex, and scanner metadata for bias testing.')


In [ ]:
from google.colab import files

print('Downloading deployment artefacts...')
files.download(os.path.join(MODEL_DIR, 'efficientnet_b0.pt'))
files.download(os.path.join(MODEL_DIR, 'config.json'))
print('\nDownloaded:')
print('  efficientnet_b0.pt  → upload to Hugging Face Space (Files tab)')
print('  config.json         → upload to Hugging Face Space (Files tab)')
